# Rasmošana ss.com - atrast dziļās saites

Ar pandas read_html mēs varam viegli iegūt datus no ss.com lapām
Bet pandas read_html last tikai tabulas, un dažreiz mums ir nepieciešams iegūt dziļākas saites vai citus elementus no lapas. Lai to izdarītu, mēs varam izmantot BeautifulSoup kopā ar requests bibliotēku.



In [1]:
# vispirms uzstādām vajadzīgās bibliotēkas
# standarta 
# vajadzēs datetime un sleep
from datetime import datetime
from time import sleep
print("Tagad ir", datetime.now())

Tagad ir 2026-09-14 21:11:17.184387


In [ ]:
# mums vajag arī ārējās bibliotekas
# requests, BeautifulSoup4, pandas
# ja nav tās jāuzstāda ar pip install requests beautifulsoup4 pandas
# ja vajag xlsx import excel, tad pip install openpyxl

import requests
from bs4 import BeautifulSoup
import pandas as pd

# izdrukāsim pandas versiju
print("Pandas versija:", pd.__version__)

Pandas versija: 3.0.5


In [3]:
# mums vajag adresi kuru rasmosim
# šoreiz īrei
URL = "https://www.ss.com/en/real-estate/flats/riga/centre/hand_over/"
print("Rasmojam adresi:", URL)

Rasmojam adresi: https://www.ss.com/en/real-estate/flats/riga/centre/hand_over/


In [4]:
# izmantojam requests lai dabūtu lapas saturu
response = requests.get(URL) # tātad šeit mēs izsaucam requests.get ar mūsu URL
# pārbaudām vai lapa atgrieza 200 OK statusu
if response.status_code == 200:
    print("Lapa atgrieza 200 OK statusu")
else:
    print("Lapa neatgrieja 200 OK statusu. Kaut kas nav labi", response.status_code)

Lapa atgrieza 200 OK statusu


In [ ]:
# mums ir pieejams texts - tātad neapstrāds html
# pirmie 200 simboli
response.text[:200] # tas ir strings tikai

'<!DOCTYPE html>\r\n<HTML lang="en"><HEAD>\r\n<title>SS.COM Flats - Riga - Centre, Prices, Hand over - Advertisements</title>\r\n<meta http-equiv="Content-Type" CONTENT="text/html; charset=UTF-8">\r\n<meta nam'

In [6]:
# noparsējam html ar BeautifulSoup
soup = BeautifulSoup(response.text, 'lxml') # vai 'html.parser' vai 'lxml' abi labi
# pārbaudam lapas nosaukumu
print("Lapas nosaukums:", soup.title)

Lapas nosaukums: <title>SS.COM Flats - Riga - Centre, Prices, Hand over - Advertisements</title>


In [8]:
# iegūsim rindu kuras id ir headline
headline = soup.find(id="head_line") # svarīgi precīzi uzrakstīt
print("Headline:", headline)

Headline: <tr id="head_line">
<td class="msg_column" colspan="3" width="70%">
<span style="float:left;"> Advertisements
</span>
<span align="right" class="msg_column" style="float:right;text-align:right;padding-right:3px;">
<noindex>
<a class="a19" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4S.html" rel="nofollow">date</a></noindex></span>
</td>
<td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4SFDwT.html" rel="nofollow" title="">Street</a></noindex></td><td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4SelM=.html" rel="nofollow" title="">R.</a></noindex></td><td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4QelM=.html" rel="nofollow" title="">m²</a></noindex></td><td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDg

In [10]:
# iegūstam visus td elementus no headline
td_elements = headline.find_all('td')
# cik    daudz td elementu ir
print("Daudz td elementu:", len(td_elements))

# izvadam katru td elementu
for i, td in enumerate(td_elements):
    print(f"TD {i}: {td}")

Daudz td elementu: 8
TD 0: <td class="msg_column" colspan="3" width="70%">
<span style="float:left;"> Advertisements
</span>
<span align="right" class="msg_column" style="float:right;text-align:right;padding-right:3px;">
<noindex>
<a class="a19" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4S.html" rel="nofollow">date</a></noindex></span>
</td>
TD 1: <td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4SFDwT.html" rel="nofollow" title="">Street</a></noindex></td>
TD 2: <td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4SelM=.html" rel="nofollow" title="">R.</a></noindex></td>
TD 3: <td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/riga/centre/hand_over/fDgSeF4QelM=.html" rel="nofollow" title="">m²</a></noindex></td>
TD 4: <td class="msg_column_td" nowrap=""><noindex><a class="a18" href="/en/real-estate/flats/r

In [11]:
# mūs intereše tikai teksts no visiem td elementiem, izņemot pirmo, jo tas ir vispārīgs
column_names = []
for td in td_elements[1:]:  # izņemot pirmo elementu
    column_names.append(td.get_text(strip=True))  # iegūstam tekstu un noņemam liekās atstarpes 
# print all columns
print("Kolonnu nosaukumi:", column_names)

Kolonnu nosaukumi: ['Street', 'R.', 'm²', 'Floor', 'Series', 'Price, m2', 'Price']


In [13]:
# tagad atradīsim visus tr elementus kuriem id sākās ar "tr_"
# vispirms tr visus
# this is oneliner
# tr_elements = soup.find_all('tr', id=lambda x: x and x.startswith('tr_')))
all_tr = soup.find_all('tr')  # atrod visus tr elementus
tr_ads = []
# ejam cauri visām rindām un paturam tikai tās kurām id sākās ar "tr_"
for tr in all_tr:
    # mumš jāpārbauda vai tr ir id un vai tas sākās ar "tr_"
    # secība ir svarīga, jo ja tr nav id tad tr.get('id') atgriezīs None un None.startswith('tr_') izraisīs kļūdu
    # ir vēl viens likums nevar sākites ar "tr_bnr"
    if tr.get('id') and tr.get('id').startswith('tr_') and not tr.get('id').startswith('tr_bnr'):
        tr_ads.append(tr)
# cik rindas atradām
print("Atrastas rindas ar id sākas ar 'tr_':", len(tr_ads))

Atrastas rindas ar id sākas ar 'tr_': 30


In [14]:
# pirmā reklāma
first_ad = tr_ads[0]
# izvadam pirmo reklāmu
print("Pirmā reklāma:", first_ad)

Pirmā reklāma: <tr id="tr_58076231"><td class="msga2 pp0"><input id="c58076231" name="mid[]" type="checkbox" value="58076231_1106_0"/></td><td class="msga2"><a href="/msg/en/real-estate/flats/riga/centre/bemphj.html" id="im58076231"><img alt="" class="isfoto foto_list" src="https://i.ss.com/gallery/8/1545/386119/77223730.th2.jpg"/></a></td><td class="msg2"><div class="d1"><a class="am" data="JTgwZiU5QmolN0ZnJUYwJTdGayU5RWx4biVBQyU4M2QlOURoemklQTQlODJkJTk3aA==|N4g8F5t" href="/msg/en/real-estate/flats/riga/centre/bemphj.html" id="dm_58076231">Izīrēju gaišu un mājīgu 2 istabu dzīvokli Rīgas centrā. 
+ noda</a></div></td><td c="1" class="msga2-o pp6" nowrap="">Pernavas 12</td><td c="1" class="msga2-o pp6" nowrap="">2</td><td c="1" class="msga2-o pp6" nowrap="">42</td><td c="1" class="msga2-o pp6" nowrap="">1/5</td><td c="1" class="msga2-o pp6" nowrap="">Chrusch.</td><td c="1" class="msga2-o pp6" nowrap="">10 €</td><td c="1" class="msga2-o pp6" nowrap="">420  €/mon.</td></tr>


In [15]:
# tātad pirmais td mūs neintereše, jo tas ir checkbox
# otrā ir a elements kur ir mūs interesējoš href
# parējos mūs interesē teksts
first_url = first_ad.find_all('td')[1].find('a').get('href')
print("Pirmās reklāmas URL:", first_url)
first_text = first_ad.find_all('td')[2].get_text(strip=True)
print("Pirmās reklāmas teksts:", first_text)

Pirmās reklāmas URL: /msg/en/real-estate/flats/riga/centre/bemphj.html
Pirmās reklāmas teksts: Izīrēju gaišu un mājīgu 2 istabu dzīvokli Rīgas centrā. 
+ noda


In [16]:
# we need BASE_URL to make full URL
BASE_URL = "https://www.ss.com"
first_full_url = BASE_URL + first_url
print("Pirmās reklāmas pilnais URL:", first_full_url)

Pirmās reklāmas pilnais URL: https://www.ss.com/msg/en/real-estate/flats/riga/centre/bemphj.html


In [17]:
# let's get text from first ad for rest of tds
texts = []
for td in first_ad.find_all('td')[2:]:  # izņemot pirmos divus elementus
    texts.append(td.get_text(strip=True))  # iegūstam tekstu un noņemam liekās atstarpes    
print(texts)

['Izīrēju gaišu un mājīgu 2 istabu dzīvokli Rīgas centrā. \r\n+ noda', 'Pernavas 12', '2', '42', '1/5', 'Chrusch.', '10 €', '420  €/mon.']


In [20]:
# saliekam to visu pirmo reklāmu kā dictionary
# pirmā atslēga mums būs url bet pārejās ņemsim no columns
first_ad_dict = {"url": first_full_url}
column_names = ["description"] + column_names  # example column names, replace with actual ones if needed
for col_name, text in zip(column_names, texts):
    first_ad_dict[col_name] = text
print("Pirmās reklāmas dati kā dictionary:")
for key,value in first_ad_dict.items():
    print(f"{key}: {value}")

Pirmās reklāmas dati kā dictionary:
url: https://www.ss.com/msg/en/real-estate/flats/riga/centre/bemphj.html
description: Izīrēju gaišu un mājīgu 2 istabu dzīvokli Rīgas centrā. 
+ noda
Street: Pernavas 12
R.: 2
m²: 42
Floor: 1/5
Series: Chrusch.
Price, m2: 10 €
Price: 420  €/mon.


In [ ]:
# tagad mums šāda rinda ir pilnībā gatava lai iebarotu pandas dataframe

In [22]:
# now let's make a loop to get all ads and make a dataframe
ads_data = []
for ad in tr_ads:
    ad_dict = {}
    ad_url = ad.find_all('td')[1].find('a').get('href')
    ad_full_url = BASE_URL + ad_url
    ad_dict["url"] = ad_full_url
    ad_texts = []
    for td in ad.find_all('td')[2:]:
        ad_texts.append(td.get_text(strip=True))
    for col_name, text in zip(column_names, ad_texts):
        ad_dict[col_name] = text
    ads_data.append(ad_dict)
# how many
print("Total ads collected:", len(ads_data))
# we a have a list of dictionaries, now let's make a dataframe

Total ads collected: 30


In [23]:
# taisam datafram no list of dictionaries
df = pd.DataFrame(ads_data)
df.to_csv("apartment_ads.csv", index=False)

In [ ]:
# TODO pašiem
# iziet cauri visām lapām un savākt visas reklāmas